In [1]:
!pip install duckdb

In [ ]:
import duckdb

### IMPORT DATA

# DuckDB (sql) is used over pandas as there were memory issues due to the dataset containing 20M+ rows, which DuckDB can handle more efficiently.

duckdb.sql("""
    COPY (
        SELECT datetime, consumption, energy_type, reading_type_name,
               site_name, site_code, postcode, organisation_name,
               organisation_type, comissioning_region, integrated_care_board,
               site_gross_internal_area, site_heated_volume,
               site_construction_year_band, site_use_type,
               electricity_reported_kwh_m2, gas_reported_kwh_m2,
               total_electrical_energy_consumption_kwh, kWh_m2
        FROM read_csv_auto('sense_data.csv')
        WHERE organisation_type LIKE '%ACUTE%'
        AND reading_type_name = '30 minute aggregated kWh'
    ) TO 'data_acute.parquet' (FORMAT PARQUET)
""")

# SENSE dataset is now filtered to only include acute hospitals and saved as a parquet file for efficient loading.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

---

## DO NOT RUN FOLLOWING CELLS (will not work)

Below are the steps for data cleaning before DuckDB was used. The following cells are now all covered during the data import using DuckDB. Explanations for removing certain columns from data are explained below:

In [ ]:
### DO NOT RUN

import polars as pl

cols_to_keep = [
    'datetime', 'consumption', 'energy_type', 'reading_type_name',
    'site_name', 'site_code', 'postcode', 'organisation_name',
    'organisation_type', 'comissioning_region', 'integrated_care_board',
    'site_gross_internal_area', 'site_heated_volume',
    'site_construction_year_band', 'site_use_type',
    'electricity_reported_kwh_m2', 'gas_reported_kwh_m2',
    'total_electrical_energy_consumption_kwh', 'kWh_m2'
]

df = pl.read_csv('data.csv', columns=cols_to_keep)
print(df.shape)
print(df.shape)

In [ ]:
### covered by DuckDB

# drop columns that are completely or almost all NaNs:

cols_to_drop = [
    'occupied_floor_area',
    'main_heating_fuel', 
    'total_thermal_energy_consumption_kwh',
    'chp_electrical_energy_consumption_kwh',
    'heat_reported_kwh_m2'
]

df = df.drop(cols_to_drop)

(10000, 26)
['datetime', 'consumption', 'reading_type', 'energy_type', 'monitor_external_reference', 'mpxn', 'site_name', 'site_code', 'postcode', 'organisation_name', 'organisation_type', 'comissioning_region', 'integrated_care_board', 'local_authority', 'site_gross_internal_area', 'site_heated_volume', 'site_construction_year_band', 'site_use_type', 'fossil_fuel_led_chp_units_operated_on_site', 'total_electrical_energy_consumption_kwh', 'chp_thermal_energy_consumption_kwh', 'electricity_reported_kwh_m2', 'gas_reported_kwh_m2', 'other_fuel_reported_kwh_m2', 'reading_type_name', 'kWh_m2']


In [ ]:
### covered by DuckDB

# further columns to drop:

cols_to_drop2 = [
    'monitor_external_reference',  # energy meter ID, not useful
    'reading_type',                # reading_type_name used to filter for actual power used
    'mpxn',                         # both identifiers for meters, not useful
    'local_authority',              # not specific to hospital and have other info on location
    'other_fuel_reported_kwh_m2',     # focusing on gas and elec and mostly NaN
    'fossil_fuel_led_chp_units_operated_on_site',  # mostly NaN
    'chp_thermal_energy_consumption_kwh'  # mostly NaN, too specific, analysis will focus on total energy consumption
]

df = df.drop(cols_to_drop2)

In [ ]:
### covered by DUCKDB


# filter to actual kWh readings only
df = df.filter(pl.col('reading_type_name') == '30 minute aggregated kWh')

# filter to acute hospitals only
df = df.filter(pl.col('organisation_type').str.contains('ACUTE'))

print(df.shape)
print(df['organisation_type'].value_counts())

(5995090, 19)


---
End of code covered by DuckDB data import, rest of data cleaning follows.
Resume running cells:

In [ ]:
import pandas as pd

# load parquet from initial data download

df = pd.read_parquet('data_acute.parquet')
print(df.shape)
print(df.head())

(5995090, 19)
                   datetime  consumption energy_type  \
0 2025-11-27 12:00:00+00:00        105.9        elec   
1 2025-11-27 12:00:00+00:00        294.8        elec   
2 2025-11-27 12:00:00+00:00          1.4        elec   
3 2025-09-15 10:00:00+00:00        212.7        elec   
4 2025-09-15 10:00:00+00:00         14.1        elec   

          reading_type_name                          site_name site_code  \
0  30 minute aggregated kWh       Homerton University Hospital     RQXM1   
1  30 minute aggregated kWh  North Manchester General Hospital     R0A66   
2  30 minute aggregated kWh              Ellen Badger Hospital     RJC04   
3  30 minute aggregated kWh              Royal United Hospital     RD130   
4  30 minute aggregated kWh                            Orbital     RN3J8   

   postcode                                  organisation_name  \
0    E9 6SR          Homerton Healthcare NHS Foundation Trust    
1    M8 5RB        Manchester University NHS Foundation Trus

In [ ]:
# recheck organisation types

print(df['organisation_type'].value_counts())

# all acute

organisation_type
ACUTE - TEACHING      3545452
ACUTE - LARGE         1467112
ACUTE - SMALL          462165
ACUTE - SPECIALIST     272869
ACUTE - MEDIUM         247492
Name: count, dtype: int64


In [ ]:
# check unique site codes and names, and commissioning regions

print(df['site_code'].nunique())
print(df['site_name'].nunique())
print(df['comissioning_region'].value_counts())

# comissioning regions are England only and do not include South East.

66
66
comissioning_region
SOUTH WEST COMMISSIONING REGION                  1714604
NORTH WEST COMMISSIONING REGION                  1331305
MIDLANDS COMMISSIONING REGION                     931730
LONDON COMMISSIONING REGION                       887999
EAST OF ENGLAND COMMISSIONING REGION              611038
NORTH EAST AND YORKSHIRE COMMISSIONING REGION     518414
Name: count, dtype: int64


In [ ]:
# check energy types

print(df['energy_type'].value_counts())

# mostly elec with some gas

energy_type
elec    4479119
gas     1515971
Name: count, dtype: int64


In [23]:
# convert to datetime and strip timezone
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_localize(None)

# set as index
df = df.set_index('datetime')

print(df.index)
print(df.index.dtype)
print(df.index.min(), df.index.max())

DatetimeIndex(['2025-11-27 12:00:00', '2025-11-27 12:00:00',
               '2025-11-27 12:00:00', '2025-09-15 10:00:00',
               '2025-09-15 10:00:00', '2025-09-15 10:00:00',
               '2025-09-15 10:00:00', '2025-09-15 10:00:00',
               '2025-09-15 10:00:00', '2025-09-15 10:00:00',
               ...
               '2025-11-27 12:00:00', '2025-11-27 12:00:00',
               '2025-11-27 12:00:00', '2025-11-27 12:00:00',
               '2025-11-27 12:00:00', '2025-11-27 12:00:00',
               '2025-11-27 12:00:00', '2025-11-27 12:00:00',
               '2025-11-27 12:00:00', '2025-11-27 12:00:00'],
              dtype='datetime64[us]', name='datetime', length=5995090, freq=None)
datetime64[us]
2022-12-07 00:00:00 2026-05-08 23:30:00


In [17]:
# count how many readings per site per day
readings_per_day = df.groupby(['site_code', 'energy_type'])['consumption'].resample('D').count()
print(readings_per_day.describe())
print(readings_per_day.value_counts().head(10))

count    72575.000000
mean        82.605443
std         94.312237
min          0.000000
25%         48.000000
50%         48.000000
75%         96.000000
max        720.000000
Name: consumption, dtype: float64
consumption
48     49868
96      9443
144     4282
192     2447
0       2135
240     1437
336     1146
720      451
576      354
624      185
Name: count, dtype: int64


In [24]:
# aggregate consumption to daily

df_daily = df.groupby(['site_code', 'site_name', 'energy_type'])['consumption'].resample('D').sum().reset_index()

# get site metadata - one row per site (static info doesn't change)
site_meta = df.groupby('site_code').first().reset_index()[['site_code', 'postcode', 
    'organisation_name', 'organisation_type', 'comissioning_region', 
    'integrated_care_board', 'site_gross_internal_area', 'site_heated_volume',
    'site_construction_year_band', 'site_use_type',
    'electricity_reported_kwh_m2', 'gas_reported_kwh_m2',
    'total_electrical_energy_consumption_kwh', 'kWh_m2']]

# merge back together
df_daily = df_daily.merge(site_meta, on='site_code', how='left')

print(df_daily.shape)
print(df_daily.columns.tolist())

(72575, 18)
['site_code', 'site_name', 'energy_type', 'datetime', 'consumption', 'postcode', 'organisation_name', 'organisation_type', 'comissioning_region', 'integrated_care_board', 'site_gross_internal_area', 'site_heated_volume', 'site_construction_year_band', 'site_use_type', 'electricity_reported_kwh_m2', 'gas_reported_kwh_m2', 'total_electrical_energy_consumption_kwh', 'kWh_m2']


In [25]:
#check time coverage per site
coverage = df_daily.groupby('site_code')['datetime'].agg(['min', 'max'])
coverage['days_covered'] = (coverage['max'] - coverage['min']).dt.days
print(coverage.sort_values('days_covered'))
print(f"\nSites with over 365 days coverage: {(coverage['days_covered'] > 365).sum()}")
print(f"Sites with over 730 days coverage: {(coverage['days_covered'] > 730).sum()}")

# 8 sites have less than 2 years of coverage.

                  min        max  days_covered
site_code                                     
R0A01      2024-02-08 2025-03-09           395
RFSUnknown 2024-01-21 2025-03-09           413
RXN16      2023-07-04 2025-03-09           614
RL1Unknown 2023-04-05 2025-03-09           704
RLT07      2023-04-05 2025-03-09           704
...               ...        ...           ...
RH8K6      2022-12-07 2026-05-08          1248
R0A06      2022-12-07 2026-05-08          1248
R0A66      2022-12-07 2026-05-08          1248
RJC46      2022-12-07 2026-05-08          1248
RKB01      2022-12-07 2026-05-08          1248

[66 rows x 3 columns]

Sites with over 365 days coverage: 66
Sites with over 730 days coverage: 58


In [26]:
# extract midpoint year from construction year band
df_daily['construction_year_mid'] = df_daily['site_construction_year_band'].str.extract(r'(\d{4})').astype(float)

# extract acute subtype
df_daily['acute_type'] = df_daily['organisation_type'].str.replace('ACUTE - ', '')

df_daily.head()

,site_code,site_name,energy_type,datetime,consumption,postcode,organisation_name,organisation_type,comissioning_region,integrated_care_board,site_gross_internal_area,site_heated_volume,site_construction_year_band,site_use_type,electricity_reported_kwh_m2,gas_reported_kwh_m2,total_electrical_energy_consumption_kwh,kWh_m2,construction_year_mid,acute_type
0,B2W3Q,Amy Johnson Way - Medical Records,elec,2024-03-28,600.600002,YO30 4AG,York And Scarborough Teaching Hospitals NHS Fo...,ACUTE - TEACHING,NORTH EAST AND YORKSHIRE COMMISSIONING REGION,NHS HUMBER AND NORTH YORKSHIRE ICB,1159.0,5793.0,1995 to 2004,Healthcare administration,144.41,325.45,167373.0,0.008628,1995.0,TEACHING
1,B2W3Q,Amy Johnson Way - Medical Records,elec,2024-03-29,474.100001,YO30 4AG,York And Scarborough Teaching Hospitals NHS Fo...,ACUTE - TEACHING,NORTH EAST AND YORKSHIRE COMMISSIONING REGION,NHS HUMBER AND NORTH YORKSHIRE ICB,1159.0,5793.0,1995 to 2004,Healthcare administration,144.41,325.45,167373.0,0.008628,1995.0,TEACHING
2,B2W3Q,Amy Johnson Way - Medical Records,elec,2024-03-30,299.700001,YO30 4AG,York And Scarborough Teaching Hospitals NHS Fo...,ACUTE - TEACHING,NORTH EAST AND YORKSHIRE COMMISSIONING REGION,NHS HUMBER AND NORTH YORKSHIRE ICB,1159.0,5793.0,1995 to 2004,Healthcare administration,144.41,325.45,167373.0,0.008628,1995.0,TEACHING
3,B2W3Q,Amy Johnson Way - Medical Records,elec,2024-03-31,441.100001,YO30 4AG,York And Scarborough Teaching Hospitals NHS Fo...,ACUTE - TEACHING,NORTH EAST AND YORKSHIRE COMMISSIONING REGION,NHS HUMBER AND NORTH YORKSHIRE ICB,1159.0,5793.0,1995 to 2004,Healthcare administration,144.41,325.45,167373.0,0.008628,1995.0,TEACHING
4,B2W3Q,Amy Johnson Way - Medical Records,elec,2024-04-01,182.800000,YO30 4AG,York And Scarborough Teaching Hospitals NHS Fo...,ACUTE - TEACHING,NORTH EAST AND YORKSHIRE COMMISSIONING REGION,NHS HUMBER AND NORTH YORKSHIRE ICB,1159.0,5793.0,1995 to 2004,Healthcare administration,144.41,325.45,167373.0,0.008628,1995.0,TEACHING


In [ ]:
# Within the 'site_use_type' column, there are some types that are not relevant to analysis of acute hospitals e.g. administrative offices. Filter the dataset to include only the following site use types:

site_types_to_keep = [
    'General acute hospital',
    'Specialist hospital (acute only)',
    'Community hospital (with inpatient beds)',
    'Mixed service hospital',
    'Other inpatient',
    'Day hospital',
    'Healthcare (general)'
]

df_daily = df_daily[df_daily['site_use_type'].isin(site_types_to_keep)]
print(f"Sites remaining: {df_daily['site_name'].nunique()}")
print(f"Rows remaining: {df_daily.shape[0]}")

### Sites for analysis: 35 acute hospitals
# unsure why output now says 42, small error in code. in next notebook, there are 35 sites.

Sites remaining: 42
Rows remaining: 50642


In [54]:
df_daily.to_parquet('sense_acute_daily.parquet', index=False)
print("Saved successfully")

Saved successfully


For now, data has been cleaned and condensed so can start from here when loading parquet in next notebook.